# Task
There is a research paper called The Quarterback Prediction Problem:
Forecasting the Performance of College
Quarterbacks Selected in the NFL Draft, that takes the tracking data from NFL football plays. It takes all the passing plays at the moment the ball is thrown and take a snapshot of defenders, and recievers. It then does analysis on it like predicting where the ball will go and what the expected reward on the throw. I like the idea but think I can improve it. First I want to recreate the results. I have given you 5 files from nfl big data bowl kaggle, your first task is summarize each one.

Then import and process the data. Create a animation that can show the player tracking data on a specific play. This is important basic step. The input should be a play id, it should then be able to make a series of snapshots of the game. Then it can play those snapshots on a image.

Start with that. The time series should have player locations, player role, like QB reciever (WR, TE, RB) linemen (Gaurds,tackle, center), secondary (LB, DB, safety), pass rush (DT, DE). The second the ball is snapped and the ball is thrown should be labeled as well.

Once we have all that we can move onto another step.

Here is all the data you need:
"tracking_week_1.csv"
"player_play.csv"
"players.csv"
"plays.csv"
"games.csv"

## Data loading

### Subtask:
Load the five provided CSV files into pandas DataFrames.


**Reasoning**:
Load the five CSV files into pandas DataFrames and display the first few rows of each to verify.



In [1]:
import pandas as pd

try:
    df_tracking = pd.read_csv('../data/tracking_week_1.csv', low_memory=False)
    display(df_tracking.head())
    df_player_play = pd.read_csv('../data/player_play.csv', low_memory=False)
    display(df_player_play.head())
    df_players = pd.read_csv('../data/players.csv', low_memory=False)
    display(df_players.head())
    df_plays = pd.read_csv('../data/plays.csv', low_memory=False)
    display(df_plays.head())
    df_games = pd.read_csv('../data/games.csv', low_memory=False)
    display(df_games.head())
except FileNotFoundError:
    print("One or more CSV files not found.")
except pd.errors.ParserError:
    print("Error parsing one or more CSV files.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

,gameId,playId,nflId,displayName,frameId,frameType,time,jerseyNumber,club,playDirection,x,y,s,a,dis,o,dir,event
0,2022091200,64,35459.0,Kareem Jackson,1,BEFORE_SNAP,2022-09-13 00:16:03.5,22.0,DEN,right,51.06,28.55,0.72,0.37,0.07,246.17,68.34,huddle_break_offense
1,2022091200,64,35459.0,Kareem Jackson,2,BEFORE_SNAP,2022-09-13 00:16:03.6,22.0,DEN,right,51.13,28.57,0.71,0.36,0.07,245.41,71.21,NaN
2,2022091200,64,35459.0,Kareem Jackson,3,BEFORE_SNAP,2022-09-13 00:16:03.7,22.0,DEN,right,51.20,28.59,0.69,0.23,0.07,244.45,69.90,NaN
3,2022091200,64,35459.0,Kareem Jackson,4,BEFORE_SNAP,2022-09-13 00:16:03.8,22.0,DEN,right,51.26,28.62,0.67,0.22,0.07,244.45,67.98,NaN
4,2022091200,64,35459.0,Kareem Jackson,5,BEFORE_SNAP,2022-09-13 00:16:03.9,22.0,DEN,right,51.32,28.65,0.65,0.34,0.07,245.74,62.83,NaN


,gameId,playId,nflId,teamAbbr,hadRushAttempt,rushingYards,hadDropback,passingYards,sackYardsAsOffense,hadPassReception,...,wasRunningRoute,routeRan,blockedPlayerNFLId1,blockedPlayerNFLId2,blockedPlayerNFLId3,pressureAllowedAsBlocker,timeToPressureAllowedAsBlocker,pff_defensiveCoverageAssignment,pff_primaryDefensiveCoverageMatchupNflId,pff_secondaryDefensiveCoverageMatchupNflId
0,2022090800,56,35472,BUF,0,0,0,0,0,0,...,NaN,NaN,47917.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1,2022090800,56,42392,BUF,0,0,0,0,0,0,...,NaN,NaN,47917.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
2,2022090800,56,42489,BUF,0,0,0,0,0,1,...,1.0,IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022090800,56,44875,BUF,0,0,0,0,0,0,...,NaN,NaN,43335.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
4,2022090800,56,44985,BUF,0,0,0,0,0,0,...,1.0,OUT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,nflId,height,weight,birthDate,collegeName,position,displayName
0,25511,6-4,225,1977-08-03,Michigan,QB,Tom Brady
1,29550,6-4,328,1982-01-22,Arkansas,T,Jason Peters
2,29851,6-2,225,1983-12-02,California,QB,Aaron Rodgers
3,30842,6-6,267,1984-05-19,UCLA,TE,Marcedes Lewis
4,33084,6-4,217,1985-05-17,Boston College,QB,Matt Ryan


,gameId,playId,playDescription,quarter,down,yardsToGo,possessionTeam,defensiveTeam,yardlineSide,yardlineNumber,...,yardsGained,homeTeamWinProbabilityAdded,visitorTeamWinProbilityAdded,expectedPointsAdded,isDropback,pff_runConceptPrimary,pff_runConceptSecondary,pff_runPassOption,pff_passCoverage,pff_manZone
0,2022102302,2655,(1:54) (Shotgun) J.Burrow pass short middle to...,3,1,10,CIN,ATL,CIN,21,...,9,0.004634,-0.004634,0.702717,True,NaN,NaN,0,Cover-3,Zone
1,2022091809,3698,(2:13) (Shotgun) J.Burrow pass short right to ...,4,1,10,CIN,DAL,CIN,8,...,4,0.002847,-0.002847,-0.240509,True,NaN,NaN,0,Quarters,Zone
2,2022103004,3146,(2:00) (Shotgun) D.Mills pass short right to D...,4,3,12,HOU,TEN,HOU,20,...,6,0.000205,-0.000205,-0.218480,True,NaN,NaN,0,Quarters,Zone
3,2022110610,348,(9:28) (Shotgun) P.Mahomes pass short left to ...,1,2,10,KC,TEN,TEN,23,...,4,-0.001308,0.001308,-0.427749,True,NaN,NaN,0,Quarters,Zone
4,2022102700,2799,(2:16) (Shotgun) L.Jackson up the middle to TB...,3,2,8,BAL,TB,TB,27,...,-1,0.027141,-0.027141,-0.638912,False,MAN,READ OPTION,0,Cover-1,Man


,gameId,season,week,gameDate,gameTimeEastern,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore
0,2022090800,2022,1,9/8/2022,20:20:00,LA,BUF,10,31
1,2022091100,2022,1,9/11/2022,13:00:00,ATL,NO,26,27
2,2022091101,2022,1,9/11/2022,13:00:00,CAR,CLE,24,26
3,2022091102,2022,1,9/11/2022,13:00:00,CHI,SF,19,10
4,2022091103,2022,1,9/11/2022,13:00:00,CIN,PIT,20,23


In [2]:
df_tracking["event"].unique()

array(['huddle_break_offense', nan, 'line_set', 'man_in_motion',
       'ball_snap', 'handoff', 'first_contact', 'tackle', 'pass_forward',
       'pass_arrived', 'pass_outcome_caught', 'touchdown', 'dropped_pass',
       'pass_outcome_incomplete', 'play_action', 'out_of_bounds', 'run',
       'qb_sack', 'pass_tipped', 'fumble', 'fumble_offense_recovered',
       'shift', 'fumble_defense_recovered', 'touchback', 'qb_strip_sack',
       'qb_kneel', 'timeout_away', 'snap_direct',
       'pass_outcome_interception', 'qb_slide', 'play_submit',
       'pass_outcome_touchdown', 'lateral', 'qb_spike', 'pass_shovel',
       'run_pass_option', 'huddle_start_offense'], dtype=object)

In [3]:
df_players.position.unique()

array(['QB', 'T', 'TE', 'WR', 'DE', 'NT', 'SS', 'FS', 'G', 'OLB', 'DT',
       'CB', 'RB', 'C', 'ILB', 'MLB', 'FB', 'DB', 'LB'], dtype=object)

## Data exploration

### Subtask:
Explore each of the five dataframes to understand their contents.


**Reasoning**:
I need to explore the dataframes to understand their contents, including data types, missing values, descriptive statistics, and unique values for categorical columns.



In [4]:
# # Explore df_tracking
# display(df_tracking.shape)
# display(df_tracking.info())
# display(df_tracking.describe())
# display(df_tracking.isnull().sum())
# for col in ['frameType', 'club', 'playDirection', 'event']:
#     display(df_tracking[col].value_counts())

# # Explore df_player_play
# display(df_player_play.shape)
# display(df_player_play.info())
# display(df_player_play.describe())
# display(df_player_play.isnull().sum())
# for col in ['teamAbbr']:
#     display(df_player_play[col].value_counts())

# # Explore df_players
# display(df_players.shape)
# display(df_players.info())
# display(df_players.describe())
# display(df_players.isnull().sum())
# for col in ['position', 'collegeName']:
#     display(df_players[col].value_counts())

# # Explore df_plays
# display(df_plays.shape)
# display(df_plays.info())
# display(df_plays.describe())
# display(df_plays.isnull().sum())
# for col in ['possessionTeam', 'defensiveTeam', 'passResult', 'offenseFormation']:
#     display(df_plays[col].value_counts())

# # Explore df_games
# display(df_games.shape)
# display(df_games.info())
# display(df_games.describe())
# display(df_games.isnull().sum())
# for col in ['homeTeamAbbr', 'visitorTeamAbbr']:
#   display(df_games[col].value_counts())

In [5]:
df_tracking["event"].unique()

array(['huddle_break_offense', nan, 'line_set', 'man_in_motion',
       'ball_snap', 'handoff', 'first_contact', 'tackle', 'pass_forward',
       'pass_arrived', 'pass_outcome_caught', 'touchdown', 'dropped_pass',
       'pass_outcome_incomplete', 'play_action', 'out_of_bounds', 'run',
       'qb_sack', 'pass_tipped', 'fumble', 'fumble_offense_recovered',
       'shift', 'fumble_defense_recovered', 'touchback', 'qb_strip_sack',
       'qb_kneel', 'timeout_away', 'snap_direct',
       'pass_outcome_interception', 'qb_slide', 'play_submit',
       'pass_outcome_touchdown', 'lateral', 'qb_spike', 'pass_shovel',
       'run_pass_option', 'huddle_start_offense'], dtype=object)

In [6]:
df_player_play[(df_player_play["gameId"] == 2022091200) & (df_player_play["playId"] == 85)]["wasTargettedReceiver"]

40502    0
40503    0
40504    0
40505    0
40506    0
40507    0
40508    0
40509    0
40510    0
40511    0
40512    0
40513    0
40514    0
40515    0
40516    0
40517    0
40518    0
40519    0
40520    1
40521    0
40522    0
40523    0
Name: wasTargettedReceiver, dtype: int64

In [7]:
df_tracking[df_tracking["event"] == 'run_pass_option'].head(2)

,gameId,playId,nflId,displayName,frameId,frameType,time,jerseyNumber,club,playDirection,x,y,s,a,dis,o,dir,event
5514979,2022091102,1726,38868.0,Tashaun Gipson,109,AFTER_SNAP,2022-09-11 18:14:13.6,31.0,SF,right,55.97,22.45,1.65,1.54,0.16,291.28,218.19,run_pass_option
5515121,2022091102,1726,43345.0,Cody Whitehair,109,AFTER_SNAP,2022-09-11 18:14:13.6,65.0,CHI,right,42.93,29.68,2.98,0.79,0.31,64.62,130.64,run_pass_option


In [8]:
df_plays[(df_plays["gameId"] == 2022091200) & (df_plays["playId"] == 85) ]["playDescription"]

8147    (14:17) (Shotgun) G.Smith pass short right to ...
Name: playDescription, dtype: object

In [9]:
# df_plays[df_plays["passResult"].isin(['C', 'I', 'IN', 'R', 'S'])]
df_tracking["event"].unique()


array(['huddle_break_offense', nan, 'line_set', 'man_in_motion',
       'ball_snap', 'handoff', 'first_contact', 'tackle', 'pass_forward',
       'pass_arrived', 'pass_outcome_caught', 'touchdown', 'dropped_pass',
       'pass_outcome_incomplete', 'play_action', 'out_of_bounds', 'run',
       'qb_sack', 'pass_tipped', 'fumble', 'fumble_offense_recovered',
       'shift', 'fumble_defense_recovered', 'touchback', 'qb_strip_sack',
       'qb_kneel', 'timeout_away', 'snap_direct',
       'pass_outcome_interception', 'qb_slide', 'play_submit',
       'pass_outcome_touchdown', 'lateral', 'qb_spike', 'pass_shovel',
       'run_pass_option', 'huddle_start_offense'], dtype=object)

In [10]:
# self.pre_snap_events = ['huddle_start_offense','huddle_break_offense', 'line_set', 'man_in_motion', 'shift']
# self.snap_events = ['ball_snap', 'snap_direct']
# self.pass_events = ['pass_forward','pass_shovel']
# self.play_end_events = ['tackle', 'touchdown', 'pass_outcome_incomplete','out_of_bounds','qb_sack', 'touchback', 'qb_kneel', 'play_submit','qb_spike',]
# self.run_events = ['handoff']
# self.post_snap_events = [ 'run_pass_option','pass_arrived', 'pass_outcome_caught',  'first_contact', 'dropped_pass', 'play_action', 'run', 'pass_tipped', 'fumble', 'fumble_offense_recovered', 'fumble_defense_recovered','qb_strip_sack', 'lateral']

# self.predict_events = ['pass_forward','run', 'qb_sack', 'pass_outcome_incomplete', 'pass_outcome_caught', 'pass_tipped', 'pass_outcome_interception', 'pass_outcome_touchdown']

In [11]:
import pandas as pd

def get_event_sequence(df_tracking, play_index):
    # Step 1: Get unique (gameId, playId) pairs
    unique_plays = df_tracking[["gameId", "playId"]].drop_duplicates().reset_index(drop=True)

    # Step 2: Get the specific (gameId, playId) for the requested index
    if play_index >= len(unique_plays):
        raise IndexError("play_index is out of range")

    game_id, play_id = unique_plays.iloc[play_index]

    # Step 3: Filter the tracking data for just this play
    play_df = df_tracking[(df_tracking["gameId"] == game_id) & (df_tracking["playId"] == play_id) & (df_tracking["displayName"] == "football")]

    # Step 4: Drop NaNs and get event sequence
    event_sequence = play_df["event"].dropna().tolist()

    

    return event_sequence
events = get_event_sequence(df_tracking, 5)
print(events)


['huddle_break_offense', 'line_set', 'ball_snap', 'pass_forward', 'pass_arrived', 'pass_outcome_caught', 'touchdown']


In [12]:
unique_plays = df_tracking[["gameId", "playId"]].drop_duplicates().reset_index(drop=True)



game_id, play_id = unique_plays.iloc[2]
game_id = 2022091200										
play_id = 85
# Step 3: Filter the tracking data for just this play
play_df = df_tracking[(df_tracking["gameId"] == game_id) & (df_tracking["playId"] == play_id) & (df_tracking["displayName"] == "football")]
play_df["event"].dropna().tolist()

['huddle_break_offense',
 'line_set',
 'man_in_motion',
 'ball_snap',
 'pass_forward',
 'pass_arrived',
 'first_contact',
 'tackle']

In [13]:
df_tracking[df_tracking["gameId"] == 2022091200]["playId"].unique()

array([  64,   85,  109,  156,  180,  201,  264,  286,  315,  346,  375,
        401,  446,  467,  565,  601,  622,  643,  664,  688,  716,  741,
        762,  786,  810,  882,  910,  931,  983, 1004, 1028, 1057, 1092,
       1164, 1217, 1241, 1299, 1320, 1344, 1409, 1433, 1465, 1487, 1521,
       1550, 1579, 1642, 1680, 1704, 1725, 1764, 1793, 1815, 1851, 1967,
       1988, 2009, 2038, 2067, 2093, 2188, 2244, 2268, 2292, 2370, 2391,
       2479, 2500, 2522, 2546, 2591, 2613, 2667, 2688, 2712, 2750, 2779,
       2801, 2830, 2883, 2923, 2944, 2965, 3001, 3048, 3077, 3101, 3125,
       3149, 3173, 3194, 3216, 3245, 3267, 3296, 3325, 3382, 3404, 3467,
       3491, 3515, 3553, 3574, 3596, 3628, 3685, 3723, 3747, 3795, 3826,
       3980, 4012], dtype=int64)

In [14]:
passing_plays = df_plays[df_plays['isDropback'] == True]
week_1_games = df_tracking["gameId"].unique()

for _, play in passing_plays.iterrows():
  game_id = play['gameId']
  if game_id not in week_1_games:
    continue
  play_id = play['playId']
  print(len(df_tracking[(df_tracking["gameId"] == game_id) & (df_tracking["playId"] == play_id) & (df_tracking["displayName"] == "football")]))

131
237
163
94
117
216
144
235
160
166
138
160
170
174
231
76
140
92
135
151
162
143
191
167
202
201
186
186
129
171
123
159
171
77
289
183
189
145
131
168
158
108
185
162
198
174
44
174
201
108
183
168
76
222
182
177
82
207
181
114
166
183
116
145
115
135
223
233
146
92
181
186
200
190
207
182
178
255
179
148
77
151
157
182
129
178
182
86
120
125
167
320
163
180
171
195
169
155
163
221
163
160
142
104
199
150


KeyboardInterrupt: 

## Data wrangling

### Subtask:
Merge the five dataframes into a single comprehensive dataframe.


**Reasoning**:
Merge the five dataframes into one, handle inconsistencies and missing values.



In [31]:
# Merge df_tracking with df_player_play
df_merged = pd.merge(df_tracking, df_player_play, on=['playId', 'nflId'], how='left')

# Merge with df_players
df_merged = pd.merge(df_merged, df_players, on='nflId', how='left')

# Merge with df_plays
df_merged = pd.merge(df_merged, df_plays, on='playId', how='left')

# Merge with df_games
df_merged = pd.merge(df_merged, df_games, on='gameId', how='left')

# Handle inconsistencies and missing values
# Inspect the data types of the join keys
print(df_merged.info())

# Check for missing values after merging
print(df_merged.isnull().sum())

# Fill missing numerical values with the mean
numerical_cols = df_merged.select_dtypes(include=['number']).columns
for col in numerical_cols:
    df_merged[col].fillna(df_merged[col].mean(), inplace=True)

# Fill missing categorical values with the mode
categorical_cols = df_merged.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df_merged[col].fillna(df_merged[col].mode()[0], inplace=True)

display(df_merged.head())

MemoryError: Unable to allocate 13.9 GiB for an array with shape (49, 37960283) and data type float64

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

def create_football_field():
    """Create a football field plot."""
    fig, ax = plt.subplots(figsize=(12, 6.33))
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 53.3)
    ax.set_xlabel('Yard Line')
    ax.set_ylabel('Field Width')
    ax.grid(True)
    return fig, ax

def animate_play(play_id, df):
    """Animate player tracking data for a specific play."""
    play_data = df[df['playId'] == play_id]
    fig, ax = create_football_field()

    # Initialize player positions
    players = ax.scatter([], [], s=100, c='red')

    def update(frame):
        """Update player positions for each frame."""
        frame_data = play_data[play_data['frameId'] == frame]
        x = frame_data['x']
        y = frame_data['y']
        players.set_offsets(np.c_[x, y])
        return players,

    # Create animation
    ani = animation.FuncAnimation(fig, update, frames=play_data['frameId'].unique(),
                                  interval=100, blit=True)
    return ani

# Example usage:
play_id = 20170907000118  # Replace with the desired play ID
ani = animate_play(play_id, df_merged)
# To see the animation, run:
# plt.show()
# To save the animation to a file, run (e.g., as a GIF):
# ani.save('play_animation.gif', writer='imagemagick')

NameError: name 'df_player_play' is not defined

In [26]:
import pandas as pd

# 1) Load your raw per‐player/frame data
df = pd.read_csv("../data/play_features.csv")

# 2) Determine, for each (gameId, playId, frameId), which club is on offense
#    We assume the QB’s club is the offense
qb_clubs = (
    df[df.position == "QB"]
    .groupby(["gameId","playId","frameId"])["club"]
    .first()
    .rename("offense_club")
)
df = df.join(qb_clubs, on=["gameId","playId","frameId"])

In [27]:
# 3) Tag offense vs defense
df["side"] = (df["club"] == df["offense_club"]).map({True:"off", False:"def"})

# 4) Define your position ordering
offense_order = ["QB","C","G","T","FB","RB","TE","WR"]
defense_order = ["NT","DE","OLB","MLB","ILB","LB","CB","DB","SS","FS"]
pos_rank = {p:i for i,p in enumerate(offense_order + defense_order)}

df["pos_rank"] = df["position"].map(pos_rank)

In [28]:
# 5) Compute a “downfield” metric (higher = further downfield for offense)
#    If playDirection is 'right', we take x as is; if 'left' we negate it
df["downfield"] = df.apply(
    lambda r:  r.x  if r.playDirection=="right" else -r.x,
    axis=1
)


In [29]:
# 6) Now group by each frame and sort players by:
#    1) side (offense first)
#    2) position rank
#    3) downfield descending
def sort_and_rank(gr):
    gr = gr.sort_values(
        by=["side","pos_rank","downfield"],
        ascending=[True, True, False]
    ).copy()
    gr["player_slot"] = range(1, len(gr)+1)
    return gr

df_ranked = df.groupby(["gameId","playId","frameId"], group_keys=False).apply(sort_and_rank)

C:\Users\jesse\AppData\Local\Temp\ipykernel_34408\424328551.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_ranked = df.groupby(["gameId","playId","frameId"], group_keys=False).apply(sort_and_rank)


In [30]:

# 7) Pivot to one‐row‐per‐frame, wide
#    Choose whichever features you'd like to keep per player; here’s an example:
to_pivot = ["x","y","s","a","o","dir","jerseyNumber","position","nflId"]
wide = df_ranked.pivot_table(
    index=["gameId","playId","frameId"],
    columns="player_slot",
    values=to_pivot,
    aggfunc="first"
)



In [39]:







# flatten the multi‐index columns
wide.columns = [
    f"{feat}_{slot}"
    for feat,slot in wide.columns
]

wide = wide.reset_index()

# 8) Export
wide.to_csv("../data/frame_features_wide.csv", index=False)


In [41]:
import pandas as pd

# 1) Load raw data
df = pd.read_csv("../data/play_features.csv")

# 2) Identify offense club by the QB in each frame
qb_clubs = (
    df[df.position == "QB"]
    .groupby(["gameId","playId","frameId"])["club"]
    .first()
    .rename("offense_club")
)
df = df.join(qb_clubs, on=["gameId","playId","frameId"])

# 3) Label offense vs defense
df["side"] = (df["club"] == df["offense_club"]).map({True:0, False:1})

# 4) Position ordering
offense_order = ["QB","C","G","T","FB","RB","TE","WR"]
defense_order = ["NT","DE","OLB","MLB","ILB","LB","CB","DB","SS","FS"]
pos_rank = {p:i for i,p in enumerate(offense_order + defense_order)}
df["pos_rank"] = df["position"].map(pos_rank)

# 5) Downfield metric
df["downfield"] = df.apply(
    lambda r:  r.x  if r.playDirection=="right" else -r.x,
    axis=1
)

# 6) Sort within each frame and assign slot
def sort_and_slot(gr):
    gr = gr.sort_values(
        by=["side","pos_rank","downfield"],
        ascending=[True, True, False]
    ).copy()
    gr["slot"] = range(1, len(gr)+1)
    return gr

df = df.groupby(
    ["gameId","playId","frameId"], 
    group_keys=False
).apply(sort_and_slot)

# 7) Select the per‐player fields you want, including name & position
player_fields = [
    ("displayName_x", "name"),
    ("position",    "pos"),
    ("x",           "x"),
    ("y",           "y"),
    ("s",           "s"),
    ("a",           "a"),
    ("o", "orientation"),
    ("dir", "dir"),

    
    ("jerseyNumber","jersey")
]
# 8) Pivot to wide format
wide = df.pivot_table(
    index=["gameId","playId","frameId"],
    columns="slot",
    values=[orig for orig,_ in player_fields],
    aggfunc="first"
)

# 9) Flatten columns in the desired order
new_cols = []
for slot in sorted(wide.columns.levels[1]):
    for orig, short in player_fields:
        new_cols.append(f"{short}_{slot}")

wide.columns = new_cols
wide = wide.reset_index()

# 10) Export
wide.to_csv("../data/frame_features_wide.csv", index=False)


C:\Users\jesse\AppData\Local\Temp\ipykernel_33212\3685680354.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(


In [33]:
import pandas as pd

# 1) Load raw data
df = pd.read_csv("../data/play_features.csv")

# 2) Identify offense club by the QB in each frame
qb_clubs = (
    df[df.position == "QB"]
    .groupby(["gameId", "playId", "frameId"])["club"]
    .first()
    .rename("offense_club")
)
df = df.join(qb_clubs, on=["gameId", "playId", "frameId"])

# 3) Label side
df["side"] = (df["club"] == df["offense_club"]).map({True: 0, False: 1})

# 4) Assign consistent role-based slot
# Stable role-based ordering
offense_roles = ["QB", "C", "G", "T", "FB", "RB", "TE", "WR"]
defense_roles = ["NT", "DT", "DE", "OLB", "MLB", "ILB", "LB", "CB", "DB", "SS", "FS"]

# Assign a consistent rank even to unknown positions
pos_rank = {p: i for i, p in enumerate(offense_roles + defense_roles)}
df["pos_rank"] = df["position"].map(pos_rank).fillna(999)

# Assign slot per frame, sorted by side, pos_rank, then x (field position)
def assign_slots(group):
    sorted_group = group.sort_values(by=["side", "pos_rank", "x"])
    sorted_group["slot"] = range(1, len(sorted_group) + 1)
    return sorted_group

df = df.groupby(["gameId", "playId", "frameId"], group_keys=False).apply(assign_slots)

# 5) Select fields to pivot
player_fields = [
    ("displayName_x", "name"),
    ("position", "pos"),
    ("x", "x"),
    ("y", "y"),
    ("s", "s"),
    ("a", "a"),
    ("o", "orientation"),
    ("jerseyNumber", "jersey")
]

# 6) Pivot
wide = df.pivot_table(
    index=["gameId", "playId", "frameId"],
    columns="slot",
    values=[orig for orig, _ in player_fields],
    aggfunc="first"
)




C:\Users\jesse\AppData\Local\Temp\ipykernel_34408\854347632.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["gameId", "playId", "frameId"], group_keys=False).apply(assign_slots)


ValueError: Length mismatch: Expected axis has 182 elements, new values have 184 elements

In [36]:
# 7) Flatten column names
# 7) Flatten column names dynamically based on actual columns
new_cols = []
actual_levels = wide.columns.levels
used_slots = wide.columns.levels[1]  # Existing slot numbers
used_fields = wide.columns.levels[0]  # Existing original column names (from player_fields)

# Only include columns that exist
for slot in used_slots:
    for orig, short in player_fields:
        if orig in used_fields:
            col_tuple = (orig, slot)
            if col_tuple in wide.columns:
                new_cols.append(f"{short}_{slot}")

# Assign only if lengths match
if len(wide.columns) != len(new_cols):
    raise ValueError(f"Column mismatch: got {len(wide.columns)} columns, but {len(new_cols)} names")

wide.columns = new_cols

# 8) Save to CSV
wide.to_csv("../data/frame_features_wide_fixed.csv", index=False)

,gameId,playId,nflId,displayName_x,frameId,frameType,time,jerseyNumber,club,playDirection,...,dis,o,dir,event,position,displayName_y,offense_club,side,pos_rank,slot
592813,2022090800,56,46076.0,Josh Allen,146,SNAP,2022-09-09 00:24:02.7,17.0,BUF,left,...,0.01,268.72,287.21,ball_snap,QB,Josh Allen,BUF,0,0.0,1
592805,2022090800,56,42392.0,Mitch Morse,146,SNAP,2022-09-09 00:24:02.7,60.0,BUF,left,...,0.01,280.26,82.19,ball_snap,C,Mitch Morse,BUF,0,1.0,2
592819,2022090800,56,48512.0,Ryan Bates,146,SNAP,2022-09-09 00:24:02.7,71.0,BUF,left,...,0.01,245.75,88.82,ball_snap,C,Ryan Bates,BUF,0,1.0,3
592802,2022090800,56,35472.0,Rodger Saffold,146,SNAP,2022-09-09 00:24:02.7,76.0,BUF,left,...,0.01,276.10,79.57,ball_snap,G,Rodger Saffold,BUF,0,2.0,4
592811,2022090800,56,44875.0,Dion Dawkins,146,SNAP,2022-09-09 00:24:02.7,73.0,BUF,left,...,0.00,273.03,17.04,ball_snap,T,Dion Dawkins,BUF,0,3.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538273,2022091200,3826,42827.0,Justin Coleman,119,AFTER_SNAP,2022-09-13 03:05:52.6,28.0,SEA,left,...,0.60,111.98,172.38,pass_forward,CB,Justin Coleman,DEN,1,15.0,19
538279,2022091200,3826,46711.0,Ryan Neal,119,AFTER_SNAP,2022-09-13 03:05:52.6,26.0,SEA,left,...,0.48,115.07,278.68,pass_forward,SS,Ryan Neal,DEN,1,17.0,20
538272,2022091200,3826,42543.0,Quandre Diggs,119,AFTER_SNAP,2022-09-13 03:05:52.6,6.0,SEA,left,...,0.37,34.73,261.58,pass_forward,FS,Quandre Diggs,DEN,1,18.0,21
538277,2022091200,3826,44873.0,Josh Jones,119,AFTER_SNAP,2022-09-13 03:05:52.6,13.0,SEA,left,...,0.17,120.54,283.09,pass_forward,FS,Josh Jones,DEN,1,18.0,22


In [15]:
import pandas as pd

# Load input features
df = pd.read_csv("../data/play_features.csv") # 5688935

In [18]:
# df.drop_duplicates() #5688080
df

gameId  \
2022091104 3662 37266.0 Jason Kelce     106 SNAP       2022-09-11 19:47:01.7    62.0   
                39950.0 Lane Johnson    106 SNAP       2022-09-11 19:47:01.7    65.0   
                43368.0 Isaac Seumalo   106 SNAP       2022-09-11 19:47:01.7    56.0   
                44834.0 Charles Harris  106 SNAP       2022-09-11 19:47:01.7    53.0   
                44888.0 Alex Anzalone   106 SNAP       2022-09-11 19:47:01.7    34.0   
...                                                                              ...   
2022110604 1051 53430.0 Trevor Lawrence 126 AFTER_SNAP 2022-11-06 18:45:48.7    16.0   
                53454.0 Travis Etienne  126 AFTER_SNAP 2022-11-06 18:45:48.7     1.0   
                53472.0 Trevon Moehrig  126 AFTER_SNAP 2022-11-06 18:45:48.7    25.0   
                54530.0 Luke Fortner    126 AFTER_SNAP 2022-11-06 18:45:48.7    79.0   
                NaN     football        126 AFTER_SNAP 2022-11-06 18:45:48.7     NaN   

                                                                                playId  \
2022091104 3662 37266.0 Jason Kelce     106 SNAP       2022-09-11 19:47:01.7       PHI   
                39950.0 Lane Johnson    106 SNAP       2022-09-11 19:47:01.7       PHI   
                43368.0 Isaac Seumalo   106 SNAP       2022-09-11 19:47:01.7       PHI   
                44834.0 Charles Harris  106 SNAP       2022-09-11 19:47:01.7       DET   
                44888.0 Alex Anzalone   106 SNAP       2022-09-11 19:47:01.7       DET   
...                                                                                ...   
2022110604 1051 53430.0 Trevor Lawrence 126 AFTER_SNAP 2022-11-06 18:45:48.7       JAX   
                53454.0 Travis Etienne  126 AFTER_SNAP 2022-11-06 18:45:48.7       JAX   
                53472.0 Trevon Moehrig  126 AFTER_SNAP 2022-11-06 18:45:48.7        LV   
                54530.0 Luke Fortner    126 AFTER_SNAP 2022-11-06 18:45:48.7       JAX   
                NaN     football        126 AFTER_SNAP 2022-11-06 18:45:48.7  football   

                                                                             frameId  \
2022091104 3662 37266.0 Jason Kelce     106 SNAP       2022-09-11 19:47:01.7   right   
                39950.0 Lane Johnson    106 SNAP       2022-09-11 19:47:01.7   right   
                43368.0 Isaac Seumalo   106 SNAP       2022-09-11 19:47:01.7   right   
                44834.0 Charles Harris  106 SNAP       2022-09-11 19:47:01.7   right   
                44888.0 Alex Anzalone   106 SNAP       2022-09-11 19:47:01.7   right   
...                                                                              ...   
2022110604 1051 53430.0 Trevor Lawrence 126 AFTER_SNAP 2022-11-06 18:45:48.7    left   
                53454.0 Travis Etienne  126 AFTER_SNAP 2022-11-06 18:45:48.7    left   
                53472.0 Trevon Moehrig  126 AFTER_SNAP 2022-11-06 18:45:48.7    left   
                54530.0 Luke Fortner    126 AFTER_SNAP 2022-11-06 18:45:48.7    left   
                NaN     football        126 AFTER_SNAP 2022-11-06 18:45:48.7    left   

                                                                              nflId  \
2022091104 3662 37266.0 Jason Kelce     106 SNAP       2022-09-11 19:47:01.7  44.70   
                39950.0 Lane Johnson    106 SNAP       2022-09-11 19:47:01.7  43.87   
                43368.0 Isaac Seumalo   106 SNAP       2022-09-11 19:47:01.7  44.13   
                44834.0 Charles Harris  106 SNAP       2022-09-11 19:47:01.7  45.77   
                44888.0 Alex Anzalone   106 SNAP       2022-09-11 19:47:01.7  45.88   
...                                                                             ...   
2022110604 1051 53430.0 Trevor Lawrence 126 AFTER_SNAP 2022-11-06 18:45:48.7  85.78   
                53454.0 Travis Etienne  126 AFTER_SNAP 2022-11-06 18:45:48.7  81.82   
                53472.0 Trevon Moehrig  126 AFTER_SNAP 2022-11-06 18:45:48.7  59.11   
             

In [ ]:
import pandas as pd

# Load input features
df = pd.read_csv("../data/play_features.csv")

new_names = ['gameId', 'playId', 'nflId','displayName','frameId', 'time',  'club', 'play_direction', 'x', 'y', 's', 'a', 'dis', 'o', 'dir', 'frame_type', 'pos', 'displayName_y']

# Renaming columns
df.columns = new_names

# Identify offense club via QB for each frame
qb_clubs = (
    df[df["position"] == "QB"]
    .groupby(["gameId", "playId", "frameId"])["club"]
    .first()
    .rename("offense_club")
)
df = df.join(qb_clubs, on=["gameId", "playId", "frameId"])

# Label offense/defense
df["side"] = (df["club"] == df["offense_club"]).map({True: 0, False: 1})

# Define role-based position ranks
offense_roles = ["QB", "C", "G", "T", "FB", "RB", "TE", "WR"]
defense_roles = ["NT", "DT", "DE", "OLB", "MLB", "ILB", "LB", "CB", "DB", "SS", "FS"]
pos_rank = {pos: i for i, pos in enumerate(offense_roles + defense_roles)}
df["pos_rank"] = df["position"].map(pos_rank).fillna(999)

# Sorting helper
def sort_players(frame):
    return frame.sort_values(
        by=["side", "pos_rank", "nflId"], ascending=[True, True, True]
    )

# Keep only player-level features (drop repeated frame-level ones)
frame_level_cols = ["gameId", "playId", "frameId"]
player_feature_cols = [col for col in df.columns if col not in frame_level_cols + ["offense_club", "side", "pos_rank", "time", "frameType", "jerseyNumber", "club", "playDirection", "event"]]

rows = []
row_ids = []
column_names = []

# Process each frame
for (gameId, playId, frameId), frame in df.groupby(frame_level_cols):
    sorted_frame = sort_players(frame)
    frame_data = sorted_frame[player_feature_cols].values.flatten()
    rows.append(frame_data)
    row_ids.append((gameId, playId, frameId))

    # Create column headers only once
    if not column_names:
        for i in range(len(sorted_frame)):
            for col in player_feature_cols:
                column_names.append(f"player{i}_{col}")

# Build final DataFrame
wide_df = pd.DataFrame(rows, columns=column_names)
wide_df.insert(0, "frameId", [fid for _, _, fid in row_ids])
wide_df.insert(0, "playId", [pid for _, pid, _ in row_ids])
wide_df.insert(0, "gameId", [gid for gid, _, _ in row_ids])

# Save
wide_df.to_csv("../data/wide_features.csv", index=False)
print(f"Saved wide_features.csv with shape: {wide_df.shape}")


KeyError: 'Column not found: club'